## Código copiado e alterado apenas o necessário do projeto de algoritmos lineares com a mesma base

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    roc_auc_score,
    roc_curve,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# estudo árvores
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

from src.config import DADOS_LIMPOS
from src.models import RANDOM_STATE

sns.set_theme(palette="bright", styly="white")

In [ ]:
df = pd.read_parquet(DADOS_LIMPOS)

df.head()

In [ ]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

In [ ]:
y.head()

In [ ]:
le = LabelEncoder()

y = le.fit_transform(y)

y[:10]

In [ ]:
le.classes_

In [ ]:
y[:20]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

https://numpy.org/doc/stable/reference/generated/numpy.bincount.html

In [ ]:
np.bincount(y_train) / len(y_train)

In [ ]:
np.bincount(y_test) / len(y_test)

In [ ]:
# mudar nosso classificador
clf = Perceptron()

clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
clf.score(X_test, y_test)

O `score` do classificador retorna a acurácia do modelo, que é a proporção de predições corretas feitas pelo modelo. Podemos calcular a acurácia comparando as predições feitas pelo modelo com as respostas corretas. A acurácia é dada por:

$$
\text{acurácia} = \frac{\text{número de predições corretas}}{\text{número total de predições}} 
$$

Uma acurácia de 1.0 significa que o modelo fez todas as predições corretas, enquanto uma acurácia de 0.0 significa que o modelo fez todas as predições erradas.

In [ ]:
accuracy_score(y_test, y_pred)

### Matriz de confusão

A matriz de confusão é uma tabela que mostra o número de predições corretas e incorretas feitas pelo modelo. A matriz de confusão é dada por (considerando a ordem do Scikit-Learn):

$$
\begin{array}{|c|c|c|}
\hline
& \text{predição negativa} & \text{predição positiva} \\
\hline
\text{classe negativa} & TN & FP \\
\hline
\text{classe positiva} & FN & TP \\
\hline
\end{array}
$$


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, normalize="true")

### Recall

Uma outra métrica que podemos usar para avaliar a qualidade de um modelo de classificação é o recall. O recall é a proporção de exemplos da classe positiva que foram corretamente classificados pelo modelo. O recall é dado por:

$$
\text{recall} = \frac{TP}{P} = \frac{TP}{TP + FN}
$$

In [ ]:
recall_score(y_test, y_pred)

### Precision (Precisão)

A precisão é a proporção de exemplos classificados como positivos que são realmente positivos. A precisão é dada por:

$$
\text{precisão} = \frac{TP}{\text{pred }P} = \frac{TP}{TP + FP}
$$



In [ ]:
precision_score(y_test, y_pred)

### F Score

O F1 score é a média harmônica entre recall e precisão. A média harmônica é usada ao invés da média aritmética porque penaliza valores extremos. O F1 score é dado por:

$$
\text{F1 score} = 2 \times \frac{\text{precisão} \times \text{recall}}{\text{precisão} + \text{recall}}
$$

Por ser uma média harmônica, o F1 score é mais sensível a valores baixos de precisão ou recall. Por exemplo, se tivermos um modelo que tem precisão 1.0 e recall 0.0, o F1 score será 0.0. Se tivermos um modelo que tem precisão 0.0 e recall 1.0, o F1 score também será 0.0.




In [ ]:
f1_score(y_test, y_pred)

O F1 score é um caso especial da média $F_\beta$:

$$
F_\beta = (1 + \beta^2) \times \frac{\text{precisão} \times \text{recall}}{\beta^2 \times \text{precisão} + \text{recall}}
$$

Quando $\beta = 1$, temos o F1 score. Quando $\beta > 1$, o recall tem mais peso na média. Quando $\beta < 1$, a precisão tem mais peso na média.

In [ ]:
fbeta_score(y_test, y_pred, beta=0.5)

In [ ]:
fbeta_score(y_test, y_pred, beta=2)

### Curva ROC

A curva ROC (Receiver Operating Characteristic) é uma curva que mostra a taxa de verdadeiros positivos (recall) em função da taxa de falsos positivos. A curva ROC é útil para avaliar a qualidade de um classificador binário em diferentes thresholds. A área sob a curva ROC (AUC) é uma métrica que varia entre 0.0 e 1.0, onde 0.0 significa um classificador ruim e 1.0 significa um classificador perfeito.



In [ ]:
roc_auc = roc_auc_score(y_test, y_pred)
roc_auc

In [ ]:
fpr, tpr, limiares = roc_curve(y_test, y_pred)

disp = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc)
disp.plot(plot_chance_level=True)

### Curva Precision-Recall

A curva precision-recall é uma curva que mostra a precisão em função do recall. A curva precision-recall é útil para avaliar a qualidade de um classificador binário em diferentes thresholds. A área sob a curva precision-recall (AUC) é uma métrica que varia entre 0.0 e 1.0, onde 0.0 significa um classificador ruim e 1.0 significa um classificador perfeito.



In [ ]:
average_precision = average_precision_score(y_test, y_pred)
average_precision

In [ ]:
y_test.mean()

In [ ]:
precision, recall, limiares = precision_recall_curve(y_test, y_pred)

disp = PrecisionRecallDisplay(
    precision=precision,
    recall=recall,
    average_precision=average_precision,
    prevalence_pos_label=y_test.mean()
)

disp.plot(plot_chance_level=True)

A reta de nível de chance no gráfico de Precision-Recall representa a precisão esperada de um classificador aleatório que faz previsões positivas independentemente das características dos dados.

É a linha que indica o desempenho de um modelo que adivinha aleatoriamente as classes dos exemplos, seguindo a mesma prevalência da classe positiva no conjunto de dados. Serve como uma linha de base para comparar o desempenho do seu modelo. Se o seu modelo não superar essa linha, ele não está melhor do que uma classificação aleatória.

### Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Visualizações e análises específicas para árvores de decisão